# Alibi — fine-tune the lie detector

Run on **Colab** (Runtime → Change runtime type → T4 GPU) or Kaggle.

1. Run the install cell.
2. Run the data cell and upload `train.jsonl`, `val.jsonl`, `test.jsonl` from
   your local `ml/data/` (produced by `uv run alibi export-dataset`).
3. Run the training cell — it prints F1 against the always-TRUE and majority
   baselines, then the fine-tuned model.
4. Run the export cell, download `alibi-detector.zip`, and unzip it into
   `ml/model/` in the repo. The server then serves it with ONNX Runtime,
   no torch.

Targets: beat always-TRUE on F1(lie); under 100 ms per statement on the N150.
Record the numbers in `ml/MODEL_CARD.md`.


In [ ]:
!pip install -q torch transformers

In [ ]:
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)

try:
    from google.colab import files  # type: ignore

    uploaded = files.upload()  # select train.jsonl, val.jsonl, test.jsonl
    for name, content in uploaded.items():
        (DATA / name).write_bytes(content)
except ImportError:
    print("Not on Colab — put train.jsonl / val.jsonl / test.jsonl in ./data/")

print(sorted(p.name for p in DATA.glob("*.jsonl")))

In [ ]:
import json

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 3


def read_jsonl(path):
    if not path.exists():
        raise SystemExit(f"missing {path}; run `uv run alibi export-dataset` first")
    with open(path) as handle:
        return [json.loads(line) for line in handle if line.strip()]


def f1_of_lies(true_labels, predicted):
    """F1 for the positive class (1 = a lie)."""
    tp = sum(1 for t, p in zip(true_labels, predicted, strict=True) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(true_labels, predicted, strict=True) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(true_labels, predicted, strict=True) if t == 1 and p == 0)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


train_rows = read_jsonl(DATA / "train.jsonl")
val_rows = read_jsonl(DATA / "val.jsonl")
test_rows = read_jsonl(DATA / "test.jsonl") if (DATA / "test.jsonl").exists() else val_rows
print(f"{len(train_rows)} train / {len(val_rows)} val / {len(test_rows)} test")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class Claims(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        encoded = tokenizer(
            row["statement"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt",
        )
        return {key: value[0] for key, value in encoded.items()}, torch.tensor(row["label"])


train_loader = DataLoader(Claims(train_rows), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Claims(val_rows), batch_size=BATCH_SIZE)
test_loader = DataLoader(Claims(test_rows), batch_size=BATCH_SIZE)


def evaluate(model, loader):
    model.eval()
    truths, predictions = [], []
    with torch.no_grad():
        for batch, labels in loader:
            logits = model(**batch).logits
            predictions += [int(p) for p in logits.argmax(dim=-1)]
            truths += [int(t) for t in labels]
    return truths, predictions


baseline_truths = [row["label"] for row in test_rows]
print("always-TRUE F1(lie):", round(f1_of_lies(baseline_truths, [0] * len(baseline_truths)), 3))
majority = max((0, 1), key=lambda label: baseline_truths.count(label))
print("majority   F1(lie):", round(f1_of_lies(baseline_truths, [majority] * len(baseline_truths)), 3))

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    for batch, labels in train_loader:
        optimizer.zero_grad()
        loss = model(**batch, labels=labels).loss
        loss.backward()
        optimizer.step()
    truths, predictions = evaluate(model, val_loader)
    print(f"epoch {epoch + 1}: val F1(lie) = {f1_of_lies(truths, predictions):.3f}")

truths, predictions = evaluate(model, test_loader)
print(f"TEST F1(lie) = {f1_of_lies(truths, predictions):.3f}")

In [ ]:
import shutil
from pathlib import Path

OUT = Path("model")
OUT.mkdir(exist_ok=True)
model.eval()
dummy = tokenizer(
    "example statement",
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
)
torch.onnx.export(
    model,
    (dummy["input_ids"], dummy["attention_mask"]),
    "model/model.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "sequence"},
        "attention_mask": {0: "batch", 1: "sequence"},
        "logits": {0: "batch"},
    },
    opset_version=17,
)
tokenizer.save_pretrained(OUT)
shutil.make_archive("alibi-detector", "zip", OUT)
print("wrote alibi-detector.zip — unzip it into ml/model/")

try:
    from google.colab import files  # type: ignore

    files.download("alibi-detector.zip")
except ImportError:
    pass

## After downloading

```bash
unzip alibi-detector.zip -d ml/model   # needs model.onnx + tokenizer.json
uv run alibi web                       # the lie_detector tool now works
```

Then measure latency on the homelab (aim < 100 ms per statement) and fill in
`ml/MODEL_CARD.md` and `docs/results/m5.md` with the F1 numbers, the baseline
comparison and three failure examples.
